
# Exercises XP : Evaluating LLMs for Summarization



## What you will learn
- Hands-on evaluation for summarization: accuracy vs. ROUGE.
- Strengths/weaknesses of metrics and model size comparisons.
- Using Hugging Face `transformers` + `evaluate` for quick experiments.
- Data loading, sampling, preprocessing, and debugging model outputs.

**Create**: evaluation scripts, comparison tables, custom metrics, and short analyses.


In [5]:
!pip install -q rouge_score==0.1.2 evaluate datasets transformers accelerate nltk

import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
import evaluate
print('Setup complete.')

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.1 MB/s eta 0:00:00


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


Setup complete.



### Part II. Dataset loading and exploration
Preferred dataset: [abisee/cnn_dailymail](https://huggingface.co/datasets/abisee/cnn_dailymail) (map `article` -> `prompt_text`, `highlights` -> `prompt_title`).
- If you have local train/test CSVs with `prompt_text` / `prompt_title`, set the paths below.
- Otherwise, we will auto-sample a small slice from the HF dataset to keep things light.
- Show a couple of rows for a sanity check.
If HF download fails, a tiny fallback sample is used.


In [6]:
import pandas as pd
from datasets import load_dataset

# Point to your data; leave empty to use the HF cnn_dailymail sample or fallback
train_path = ''
test_path = ''

fallback = pd.DataFrame([
    {
        'prompt_text': 'The cat sat on the mat and purred loudly while the sun set.',
        'prompt_title': 'Cat rests on mat at sunset'
    },
    {
        'prompt_text': 'Scientists discovered water on the moon, opening new research paths.',
        'prompt_title': 'Water found on the moon'
    }
])

def load_and_sample(path, split_name, n):
    if path:
        return pd.read_csv(path).sample(min(n, 10), random_state=42)
    try:
        hf_split = f'{split_name}[:{n}]'
        ds = load_dataset('abisee/cnn_dailymail', '3.0.0', split=hf_split)
        return ds.to_pandas()[['article', 'highlights']].rename(columns={'article': 'prompt_text', 'highlights': 'prompt_title'})
    except:
        return fallback.copy()

train_df = load_and_sample(train_path, 'train', 100)
test_df = load_and_sample(test_path, 'test', 50)
display(train_df.head(2))

README.md:   0%|          | 0.00/15.6k [00:00<?, ?B/s]

3.0.0/train-00000-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00001-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00002-of-00003.parquet:   0%|          | 0.00/259M [00:00<?, ?B/s]

3.0.0/validation-00000-of-00001.parquet:   0%|          | 0.00/34.7M [00:00<?, ?B/s]

3.0.0/test-00000-of-00001.parquet:   0%|          | 0.00/30.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/287113 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/13368 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11490 [00:00<?, ? examples/s]

,prompt_text,prompt_title
0,"LONDON, England (Reuters) -- Harry Potter star...",Harry Potter star Daniel Radcliffe gets £20M f...
1,Editor's note: In our Behind the Scenes series...,Mentally ill inmates in Miami are housed on th...



### Part III. Summarization with T5 (implement)
Tasks:
- Write `batch_generator` to yield mini-batches.
- Write `summarize_with_t5` using `t5-small` (or swap sizes) with GPU if available.
- Prefix inputs with "summarize: " and decode with `skip_special_tokens=True`.
- Clear CUDA cache between batches (`torch.cuda.empty_cache()`) and gc.collect().


In [7]:
import torch, gc
from transformers import AutoTokenizer, T5ForConditionalGeneration
from typing import Iterable, List
import pandas as pd

def batch_generator(items: List[str], batch_size: int):
    for i in range(0, len(items), batch_size):
        yield items[i : i + batch_size]

def summarize_with_t5(texts: List[str], model_name: str = "t5-small", batch_size: int = 4, max_new_tokens: int = 32):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = T5ForConditionalGeneration.from_pretrained(model_name).to(device)

    results = []
    for batch in batch_generator(texts, batch_size):
        inputs = tokenizer(["summarize: " + t for t in batch], return_tensors="pt", padding=True, truncation=True).to(device)
        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=max_new_tokens)
        results.extend(tokenizer.batch_decode(outputs, skip_special_tokens=True))

        torch.cuda.empty_cache()
        gc.collect()
    return results

RUN_T5 = True
if RUN_T5 and 'train_df' in locals():
    sample_texts = train_df['prompt_text'].head(10).tolist()
    train_summaries_t5 = summarize_with_t5(sample_texts)
    display(pd.DataFrame({
        'prompt_text': sample_texts,
        'reference_summary': train_df['prompt_title'].head(10),
        't5_small_summary': train_summaries_t5
    }).head())
else:
    print("T5 generation skipped. Ensure train_df is defined in Part II.")

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

,prompt_text,reference_summary,t5_small_summary
0,"LONDON, England (Reuters) -- Harry Potter star...",Harry Potter star Daniel Radcliffe gets £20M f...,he says he has no plans to fritter cash away o...
1,Editor's note: In our Behind the Scenes series...,Mentally ill inmates in Miami are housed on th...,inmates with most severe mental illnesses are ...
2,"MINNEAPOLIS, Minnesota (CNN) -- Drivers who we...","NEW: ""I thought I was going to die,"" driver sa...","driver on the bridge says he had a 30-, 35-foo..."
3,WASHINGTON (CNN) -- Doctors removed five small...,"Five small polyps found during procedure; ""non...",new: president's doctor recommended repeat pro...
4,(CNN) -- The National Football League has ind...,"NEW: NFL chief, Atlanta Falcons owner critical...",new: nfl suspends quarterback without pay. new...



### Part IV. Accuracy evaluation (toy, likely near zero)
Implement a naive accuracy that checks exact string match between generated and reference summaries.
Discuss why this is harsh for free-form text (almost always zero).


In [9]:

from typing import List

def compute_accuracy(preds: List[str], refs: List[str]) -> float:
    matches = sum(1 for p, r in zip(preds, refs) if p.strip() == r.strip())
    return matches / max(len(refs), 1)

if 'train_summaries_t5' in locals():
    acc = compute_accuracy(train_summaries_t5, train_df['prompt_title'].tolist())
    print(f"Exact-match accuracy: {acc:.4f}")
else:
    print("Accuracy skipped (no predictions).")


Exact-match accuracy: 0.0000



### Part V. ROUGE metric implementation
Use `evaluate.load("rouge")` and NLTK sentence tokenizer.
Preprocess by joining sentences with newlines for better ROUGE-L.


In [10]:
import evaluate
from nltk.tokenize import sent_tokenize
from typing import List

try:
    rouge = evaluate.load('rouge')
except:
    print("Please run Part I cell to install 'evaluate'")

def normalize_text(text):
    sents = sent_tokenize(str(text).strip())
    return "\n".join(sents)

def compute_rouge_score(preds: List[str], refs: List[str]):
    norm_preds = [normalize_text(p) for p in preds]
    norm_refs = [normalize_text(r) for r in refs]
    return rouge.compute(predictions=norm_preds, references=norm_refs)

# Sanity check
test_preds = ["alpha beta"]
test_refs  = ["alpha beta"]
if 'rouge' in locals():
    print("ROUGE check:", compute_rouge_score(test_preds, test_refs))

ROUGE check: {'rouge1': np.float64(1.0), 'rouge2': np.float64(1.0), 'rougeL': np.float64(1.0), 'rougeLsum': np.float64(1.0)}



### Part VI. Understanding ROUGE scores
Experiments to run (describe your findings in a text cell):
- Exact match vs. empty prediction.
- Effect of stemming: e.g., "running" vs. "run".
- N-gram overlap: see how ROUGE-1 vs. ROUGE-2 change with partial overlap.
- Symmetry: swap preds/refs and compare.



### Part VII. Comparing small and large models
Goals:
- Generate summaries with `t5-small`, `t5-base`, and `gpt2` (TL;DR style prompt).
- Compute ROUGE for each and store per-row scores.
- Implement `compute_rouge_per_row` to add ROUGE columns to a DataFrame.
- Implement `summarize_with_gpt2` with a TL;DR: prefix and max length guard.
Use small batches and low `max_new_tokens` to keep things snappy.


In [11]:
from transformers import AutoModelForCausalLM, AutoTokenizer

def summarize_with_gpt2(texts: List[str], model_name: str = 'gpt2', batch_size: int = 2, max_new_tokens: int = 32):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

    results = []
    for batch in batch_generator(texts, batch_size):
        # Simple TL;DR prompt
        prompts = [t[:500] + "\n\nTL;DR:" for t in batch]
        inputs = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True, max_length=1024).to(device)

        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, pad_token_id=tokenizer.eos_token_id)

        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        # Extract only the generated part after TL;DR:
        extracted = [d.split("TL;DR:")[-1].strip() for d in decoded]
        results.extend(extracted)

        torch.cuda.empty_cache()
        gc.collect()
    return results

def compute_rouge_per_row(df: pd.DataFrame, pred_col: str, ref_col: str = 'prompt_title'):
    scores = []
    for _, row in df.iterrows():
        score = compute_rouge_score([row[pred_col]], [row[ref_col]])
        scores.append(score['rougeL'])
    df[f'{pred_col}_rougeL'] = scores
    return df

RUN_COMPARE = True
if RUN_COMPARE and 'train_summaries_t5' in locals():
    eval_df = train_df.head(10).copy()
    eval_df['t5_small_summary'] = train_summaries_t5
    eval_df['gpt2_summary'] = summarize_with_gpt2(eval_df['prompt_text'].tolist())

    eval_df = compute_rouge_per_row(eval_df, 't5_small_summary')
    eval_df = compute_rouge_per_row(eval_df, 'gpt2_summary')
    display(eval_df[['prompt_title', 't5_small_summary', 'gpt2_summary', 't5_small_summary_rougeL', 'gpt2_summary_rougeL']])

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

[transformers] A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
[transformers] A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
[transformers] A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
[transformers] A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
[transformers] A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.


,prompt_title,t5_small_summary,gpt2_summary,t5_small_summary_rougeL,gpt2_summary_rougeL
0,Harry Potter star Daniel Radcliffe gets £20M f...,he says he has no plans to fritter cash away o...,Harry Potter star Daniel Radcliffe has no plan...,0.322581,0.412698
1,Mentally ill inmates in Miami are housed on th...,inmates with most severe mental illnesses are ...,", the ""forgotten floor"" is where most of the m...",0.138889,0.309859
2,"NEW: ""I thought I was going to die,"" driver sa...","driver on the bridge says he had a 30-, 35-foo...","The bridge collapsed on the Mississippi.\n\n""I...",0.307692,0.156250
3,"Five small polyps found during procedure; ""non...",new: president's doctor recommended repeat pro...,", Bush's colon is not a big deal.\n\n\nThe pol...",0.142857,0.081633
4,"NEW: NFL chief, Atlanta Falcons owner critical...",new: nfl suspends quarterback without pay. new...,", Vick is set to appear in court Monday.\n\nVi...",0.280702,0.121212
5,"Parents beam with pride, can't stop from smili...","youssif, 5, is wearing a mask often used to he...",Youssif's story is a story of hope and hope.\n...,0.064516,0.063492
6,"Aid workers: Violence, increased cost of livin...",women are too afraid to show their faces or ha...,",,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,",0.157895,0.000000
7,Tomas Medina Caracas was a fugitive from a U.S...,fugitive from drug trafficking indictment kill...,A Colombian rebel commander and fugitive from ...,0.379310,0.514286
8,"President Bush says Tony Snow ""will battle can...","new: president bush says he will ""sadly accept...",", Snow will be replaced by Perino.\n\nSnow's d...",0.339623,0.166667
9,Empty anti-tank weapon turns up in front of Ne...,new: the 20-year-old device is a one-time-use ...,The rocket launcher tube was found in a home i...,0.160000,0.206897



### Part VIII. Comparing all models
Implement:
- `compare_models` to aggregate average ROUGE across models.
- `compare_models_summaries` to show side-by-side summaries.
Present the tables and discuss which model wins and why.


In [12]:
import pandas as pd

def compare_models(results_df):
    # Aggregate average ROUGE-L from the columns created earlier
    rouge_cols = [c for c in results_df.columns if '_rougeL' in c]
    return results_df[rouge_cols].mean().to_frame(name='Avg ROUGE-L')

def compare_models_summaries(df: pd.DataFrame, pred_cols: list):
    cols = ['prompt_title'] + pred_cols
    return df[cols].head(5)

if 'eval_df' in locals():
    print("Model Metrics Comparison:")
    display(compare_models(eval_df))
    print("\nSample Summaries:")
    display(compare_models_summaries(eval_df, ['t5_small_summary', 'gpt2_summary']))

Model Metrics Comparison:


,Avg ROUGE-L
t5_small_summary_rougeL,0.229406
gpt2_summary_rougeL,0.203299



Sample Summaries:


,prompt_title,t5_small_summary,gpt2_summary
0,Harry Potter star Daniel Radcliffe gets £20M f...,he says he has no plans to fritter cash away o...,Harry Potter star Daniel Radcliffe has no plan...
1,Mentally ill inmates in Miami are housed on th...,inmates with most severe mental illnesses are ...,", the ""forgotten floor"" is where most of the m..."
2,"NEW: ""I thought I was going to die,"" driver sa...","driver on the bridge says he had a 30-, 35-foo...","The bridge collapsed on the Mississippi.\n\n""I..."
3,"Five small polyps found during procedure; ""non...",new: president's doctor recommended repeat pro...,", Bush's colon is not a big deal.\n\n\nThe pol..."
4,"NEW: NFL chief, Atlanta Falcons owner critical...",new: nfl suspends quarterback without pay. new...,", Vick is set to appear in court Monday.\n\nVi..."


## Wrap-up & Reflection

### Findings
- **Metrics**: ROUGE-L proved more informative than exact accuracy because it captures semantic overlap rather than literal character matching, which is essential for creative tasks like summarization.
- **Model Comparison**: T5-small typically produces more coherent summaries than base GPT-2 (without fine-tuning) because it was pre-trained specifically for sequence-to-sequence tasks like translation and summarization.
- **Accuracy Breakdown**: Exact-match accuracy was near zero because even a single character difference (like a trailing space) causes a failure, making it useless for evaluating linguistic quality.

### Potential Extensions
To extend this work, one could implement **BERTScore** for embedding-based similarity or conduct a **Human Evaluation** where summaries are ranked on a scale of 1-5 for factuality and fluency.